# 04 — Quantum Mutual Information on Real Hardware

In `02/11` a Bell pair shared exactly 2 bits of mutual information — double the classical maximum any pair of bits can reach. That factor of two *is* entanglement, written in the language of information. Here you measure that same quantity on a real superconducting chip and watch decoherence quietly steal a piece of it.

**Prerequisites:** `01/19_entanglement`, `02/11_quantum_mutual_information`, `05/03_tomography_and_fidelity_on_real_hardware`.

**What you'll be able to do**
- Prepare a Bell pair on the device and run full two-qubit tomography (nine measurement settings).
- Reconstruct the joint state `rho_AB` from the device's counts with `density_from_counts`.
- Compute the quantum mutual information `I(A:B)` and watch it fall below the ideal 2 bits — below it because of decoherence *and* finite-shot effects together.

You have already reconstructed a single-qubit state on hardware in `05/03` and read its fidelity gap. Now you lift that machinery to two qubits and recover the *joint* state of a Bell pair — the first information-geometric quantity of this module to meet a live device. Measuring entanglement on a chip cooled to a few millikelvin, from nothing but click statistics, is a genuine milestone. Let us go and do it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.records import save_run_record
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import two_qubit_tomography_circuits, density_from_counts
from qot_course.infotheory.quantum import quantum_mutual_information
from qot_course.quantum.composite import bell_state
from qot_course.quantum.density import density_matrix, fidelity

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")
if not is_real:
    backend.set_options(seed_simulator=42)  # fixed seed: the offline numbers quoted below are reproducible

## 1 · Two-qubit tomography

One qubit needed three measurement settings — one per Pauli axis. Two qubits need the nine combinations of $\{X, Y, Z\}$ on each: $XX, XY, XZ, YX, \ldots, ZZ$. Together those nine settings pin down every entry of the $4 \times 4$ joint density matrix, exactly as the three single-qubit settings fixed the Bloch vector in `05/03`. We reuse the very same reconstruction, `density_from_counts`, now told there are two qubits.

From the reconstructed `rho_AB` we compute the quantum mutual information

$$I(A:B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB}),$$

where $S(\rho) = -\mathrm{Tr}(\rho \log_2 \rho)$ is the von Neumann entropy in bits and $\rho_A, \rho_B$ are the reduced states of each qubit. For a pure Bell pair each qubit alone is maximally mixed ($S = 1$ bit) while the joint state is pure ($S = 0$), so $I(A:B) = 1 + 1 - 0 = 2$ bits — the ceiling a real device cannot quite reach.

In [ ]:
prep = QuantumCircuit(2)
prep.h(0)
prep.cx(0, 1)                               # Bell pair (|00> + |11>)/sqrt(2)
circuits = two_qubit_tomography_circuits(prep)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
settings = list(circuits)                   # 9 (basis_q0, basis_q1) tuples
isa_list = [pm.run(circuits[s]) for s in settings]
SHOTS = 4096
job = SamplerV2(mode=backend).run(isa_list, shots=SHOTS)
res = job.result()
counts_by_setting = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}
save_run_record(counts_by_setting, notebook="05_04", tag="tomography_bell", device=backend.name, shots=SHOTS,
                job_id=job.job_id() if is_real else None, is_real=is_real)

rho_ab = density_from_counts(counts_by_setting, n_qubits=2)
ideal = density_matrix(bell_state())
qmi_device = quantum_mutual_information(rho_ab, dims=[2, 2])   # bits
qmi_ideal = quantum_mutual_information(ideal, dims=[2, 2])
print(f"fidelity(rho_AB, Bell) = {fidelity(rho_ab, ideal):.3f}")
print(f"I(A:B) ideal  = {qmi_ideal:.3f} bits")
print(f"I(A:B) device = {qmi_device:.3f} bits   (below the ideal 2 bits)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.bar(["ideal Bell", "device"], [qmi_ideal, qmi_device],
       color=[COLORS["muted"], COLORS["quantum"]], alpha=0.9)
ax.axhline(1.0, color=COLORS["highlight"], linestyle="--", label="classical maximum (1 bit)")
ax.set_ylabel("I(A:B)  (bits)")
ax.set_title("Quantum mutual information: ideal Bell pair vs a real device", pad=12)
ax.legend()
plt.show()

**Read the figure.** The ideal pair clears the classical 1-bit ceiling and reaches 2 bits — the information signature of maximal entanglement. (That reading is exact here because the Bell pair is a *pure* state: for a pure state, quantum mutual information equals twice the entanglement entropy, so the 2-bit ceiling is genuinely a statement about entanglement. For a general mixed state, QMI also counts classical correlations and is not an entanglement measure on its own — a distinction that matters once the device has already introduced noise into `rho_AB`.) The device bar falls short, for two reasons. The device leaks correlation: every two-qubit gate (a `cz` here, from which the transpiler builds your `cx`) and every read-out error loses a little of what the Bell pair stores. And the tomography itself costs something: finite shots and the projection back to a physical state inside `density_from_counts` bias the reconstructed entropy, so even a noiseless device would land below 2 bits. The next cell measures how the gap divides between the two. Offline this reflects the modeled FakeMarrakesh noise; flip `USE_HARDWARE=True` and the very same bar reports the literal chip's gap.

### Splitting the gap

How much of that shortfall is the device, and how much is the tomography itself? A noiseless simulator answers it. Run the same nine settings, with the same 4,096 shots, on a simulator with no noise at all: whatever it loses below 2 bits is the finite-shot and projection part, and the rest of the device's gap is the device's own. Ten noiseless repeats give that part with its scatter. The cell also prints the calibrated error rates of the gates and read-outs the circuit used.

In [ ]:
from qiskit_aer import AerSimulator

N_NOISELESS = 10
noiseless = AerSimulator(seed_simulator=7)   # no noise model: only finite shots and the projection remain
pm_noiseless = generate_preset_pass_manager(backend=noiseless, optimization_level=1)
nl_circuits = [pm_noiseless.run(circuits[s]) for s in settings]
nl_res = SamplerV2(mode=noiseless).run(nl_circuits * N_NOISELESS, shots=SHOTS).result()
qmi_noiseless = []
for r in range(N_NOISELESS):
    cbs = {s: nl_res[r * len(settings) + i].data.c.get_counts() for i, s in enumerate(settings)}
    qmi_noiseless.append(quantum_mutual_information(density_from_counts(cbs, n_qubits=2), dims=[2, 2]))
qmi_nl, qmi_nl_sd = float(np.mean(qmi_noiseless)), float(np.std(qmi_noiseless, ddof=1))
gap_total = qmi_ideal - qmi_device
gap_shots = qmi_ideal - qmi_nl
print(f"noiseless, {SHOTS} shots/setting: I(A:B) = {qmi_nl:.3f} +/- {qmi_nl_sd:.3f} bits over {N_NOISELESS} repeats")
print(f"total gap (ideal - device)          = {gap_total:.3f} bits")
print(f"finite shots + projection (ideal - noiseless) = {gap_shots:.3f} bits  ({gap_shots / gap_total:.0%} of the gap)")
print(f"device noise (noiseless - device)    = {qmi_nl - qmi_device:.3f} bits  ({(qmi_nl - qmi_device) / gap_total:.0%} of the gap)")

q0, q1 = isa_list[0].layout.final_index_layout()[:2]
target = backend.target
two_q = next(op for op in ("cz", "ecr", "cx") if op in target.operation_names)
pair = (q0, q1) if (q0, q1) in target[two_q] else (q1, q0)
print(f"calibrated error rates on physical qubits {q0}, {q1}:")
print(f"  two-qubit {two_q} {pair}: {target[two_q][pair].error:.4f}")
for q in (q0, q1):
    print(f"  qubit {q}: sx {target['sx'][(q,)].error:.5f}   read-out {target['measure'][(q,)].error:.4f}")

**Read the output.** On the offline model the device returns $1.790$ bits, a gap of $0.210$ bits. A noiseless simulator with the same 4,096 shots per setting returns $1.915 \pm 0.030$ bits, so finite shots and the projection account for $0.085$ bits, about 40% of the gap, and the device's noise for the other $0.125$ bits, about 60%. Take the split as a first estimate: the two effects need not add exactly, and the noiseless value scatters by $0.030$ bits from run to run. It is not the "mostly decoherence" one might guess: at 4,096 shots the tomography's own bias is a large share of what you see, and that share shrinks as you raise the shots. The error rates show where the device's share comes from. The `cz` between qubits 0 and 1 errs with probability $0.0017$, more than ten times the `sx` rotations ($0.00013$ and $0.00010$), and the two read-outs err with $0.0112$ and $0.0054$, the largest rates of all.

### A real run on IBM hardware

This notebook was run once on a real quantum computer. On **`ibm_marrakesh`** (156-qubit Heron processor, 2026-05-30, job `d8dih47d0j8c73f4apmg`), the full nine-setting two-qubit tomography returned:

| quantity | ideal | this device |
|---|---|---|
| fidelity to the Bell state | 1.00 | **0.96** |
| I(A:B) | 2.00 bits | **1.74 bits** |

A genuine measurement: `I(A:B) = 1.74` bits sits below the ideal 2 while still clearing the classical 1-bit ceiling. The offline twin of the same device gives $1.790$ bits above. Only the job identifier of that run was kept, not its raw counts, so the table cannot be recomputed from this repository. Set `USE_HARDWARE = True` to reproduce on the least-busy QPU your account can reach.

## 2 · Why entanglement is fragile

The two-qubit entangling gate (`cz` on this device) and the read-out are the noisiest operations of this circuit, as the calibrated rates above show: each errs ten to a hundred times more often than a single-qubit `sx` rotation. A Bell pair is *made of* exactly those operations: a Hadamard, an entangling gate, and a two-qubit read-out. So the correlations that make a Bell pair special are precisely what decoheres first, which is why `I(A:B)` is a more demanding probe of a device than the single-qubit fidelity of `05/03`.

This adds two rows to the ideal-versus-device table you began in `05/03`:

| Ideal object | What a real device returns |
|---|---|
| Bell pair with `I(A:B) = 2` bits | `rho_AB` with `I(A:B) < 2` — correlation leaked to the environment |
| pure, maximally entangled joint state | mixed `rho_AB` (fidelity < 1), reconstructed from finite shots |

Where `05/03` measured how far one qubit fell from a pure target, here you measure how much *shared information* two qubits lost — entanglement made quantitative as a number of bits.

## Your turn

- **Try the other Bell state.** Build the `|01> + |10>` variant with `prep.h(0); prep.cx(0, 1); prep.x(1)` and re-run. Its ideal `I(A:B)` is also 2 bits, so check whether the device's value drops by a similar amount — the entanglement loss should track the gate count, not which Bell state you chose.
- **Flip `USE_HARDWARE = True`** (with credentials set per `05/01`) and re-run. The gap below 2 bits is now the real chip's entanglement loss, not a model's.
- **Raise the shots and split the gap.** Re-run at higher shot counts (say `8192`, then `16384`) and watch the *finite-shot* part of the gap shrink while the *systematic* decoherence part stays put. This is the same statistical-versus-systematic split you met on the Born rule in `05/02` — only now it acts on an information-geometric quantity instead of a single probability.

## What you built

- A full two-qubit tomography run on the primitive path: a Bell pair prepared on the device, nine Pauli-basis circuits transpiled and sampled with `SamplerV2`, and the joint state `rho_AB` reconstructed with `density_from_counts`.
- The quantum mutual information `I(A:B)` read off that reconstructed state, the first information-geometric quantity of this module measured on a live device.
- A direct sighting of entanglement loss: `I(A:B)` falls below the ideal 2 bits, with the gap split by a noiseless simulation of the same shots: on the offline model, about 60% device noise and 40% finite-shot and projection bias, the part that shrinks with more shots.
- Two new rows of the ideal-versus-device table, tying the ideal Bell pair, with `I(A:B) = 2` bits, to the mixed `rho_AB` the device returns and its diminished mutual information.

## What's next

Next you measure **geometry** on real hardware: the **Bures distance** between the ideal and device states, turning the fidelity gap of `05/03` into a genuine metric on the Bures geometry, the geometry that modules 02 to 04 built and bridged to optimal transport.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 11 (eq. 11.78, quantum mutual information), Cambridge University Press (2010).
- M. M. Wilde, *Quantum Information Theory*, 2nd ed., ch. 11, Cambridge University Press (2017), doi:10.1017/9781316809976.

**Previous:** `notebooks/05_RealQuantumHardware/03_tomography_and_fidelity_on_real_hardware.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/05_geometry_on_real_hardware_bures.ipynb`